# Disease Prediction Model Training and Selection – Notebook Overview

This notebook focuses on training, evaluating, and selecting the most suitable machine learning model for symptom-based disease prediction. It uses a structured dataset where symptoms are represented in binary format and diseases are treated as multi-class labels.

## Model Training and Comparative Evaluation

In this notebook, the symptom dataset is processed and used to train multiple machine learning classification algorithms, including Decision Tree, Random Forest, Naive Bayes, Logistic Regression, Support Vector Machine, K-Nearest Neighbour, and XGBoost. Each model is evaluated using standard performance metrics such as accuracy, precision, recall, and weighted F1-score to ensure a fair and consistent comparison.

## Model Selection

Based on the evaluation results, the most reliable and scalable model is identified. Special attention is given to models that not only perform well on the current dataset but are also capable of handling increased feature complexity in the future. XGBoost is selected as the final model due to its robustness, ensemble learning capability, and strong performance on structured data.

## Final Model Training and Saving

After selecting the best-performing model, XGBoost is retrained using the complete dataset to maximize learning from all available data. The fully trained model, along with necessary encoders and feature mappings, is then saved for deployment in the main disease prediction system.

# Import and pre process the data

In [20]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report

In [21]:
# Load the dataset
# Assuming CSV file with 128 symptoms as features and one 'disease' column as target
#Evaluator can replace the path to the right file during evaluation
df = pd.read_csv("C://Users//kalya//Desktop//Kalyan//Upgrad AIML//Thesis Topics//dataset.csv")


In [22]:
# List of symptom columns
symptom_columns = [f'Symptom_{i}' for i in range(1, 18)]  # Symptom_1 to Symptom_17

# Flatten and get unique symptoms
all_symptoms = pd.unique(df[symptom_columns].values.ravel())
all_symptoms = [symptom for symptom in all_symptoms if pd.notnull(symptom)]  # Remove NaN

# Create binary columns for each unique symptom
for symptom in all_symptoms:
    df[symptom] = df[symptom_columns].apply(lambda row: int(symptom in row.values), axis=1)

#  Drop the original Symptom_1 to Symptom_17 columns
df = df.drop(columns=symptom_columns)

# Save to new CSV
df.to_csv("C://Users//kalya//Desktop//Kalyan//Upgrad AIML//Thesis Topics//binary_symptom_dataset.csv", index=False)

print("✅ Symptom binary encoding completed and saved to 'binary_symptom_dataset.csv'")

C:\Users\kalya\AppData\Local\Temp\ipykernel_22028\1609365322.py:10: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[symptom] = df[symptom_columns].apply(lambda row: int(symptom in row.values), axis=1)
C:\Users\kalya\AppData\Local\Temp\ipykernel_22028\1609365322.py:10: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[symptom] = df[symptom_columns].apply(lambda row: int(symptom in row.values), axis=1)
C:\Users\kalya\AppData\Local\Temp\ipykernel_22028\1609365322.py:10: PerformanceWarning: DataFrame is highly fragmented.  This is u

✅ Symptom binary encoding completed and saved to 'binary_symptom_dataset.csv'


C:\Users\kalya\AppData\Local\Temp\ipykernel_22028\1609365322.py:10: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[symptom] = df[symptom_columns].apply(lambda row: int(symptom in row.values), axis=1)
C:\Users\kalya\AppData\Local\Temp\ipykernel_22028\1609365322.py:10: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[symptom] = df[symptom_columns].apply(lambda row: int(symptom in row.values), axis=1)
C:\Users\kalya\AppData\Local\Temp\ipykernel_22028\1609365322.py:10: PerformanceWarning: DataFrame is highly fragmented.  This is u

In [26]:
df.head

<bound method NDFrame.head of               Disease  itching   skin_rash   nodal_skin_eruptions  \
0    Fungal infection        1           1                      1   
1    Fungal infection        0           1                      1   
2    Fungal infection        1           0                      1   
3    Fungal infection        1           1                      0   
4    Fungal infection        1           1                      1   
..                ...      ...         ...                    ...   
299          Impetigo        0           0                      0   
300          Impetigo        0           1                      0   
301          Impetigo        0           1                      0   
302          Impetigo        0           1                      0   
303          Impetigo        0           1                      0   

      dischromic _patches   continuous_sneezing   shivering   chills  \
0                       1                     0           0        0 

In [28]:
df.columns

Index(['Disease', 'itching', ' skin_rash', ' nodal_skin_eruptions',
       ' dischromic _patches', ' continuous_sneezing', ' shivering', ' chills',
       ' watering_from_eyes', ' stomach_pain',
       ...
       ' bladder_discomfort', ' foul_smell_of urine',
       ' continuous_feel_of_urine', ' skin_peeling', ' silver_like_dusting',
       ' small_dents_in_nails', ' inflammatory_nails', ' blister',
       ' red_sore_around_nose', ' yellow_crust_ooze'],
      dtype='object', length=132)

In [30]:

# Separate features and target
X = df.drop(columns=['Disease'])  # Symptoms
y = df['Disease']                 # Disease label


In [32]:
X.columns

Index(['itching', ' skin_rash', ' nodal_skin_eruptions',
       ' dischromic _patches', ' continuous_sneezing', ' shivering', ' chills',
       ' watering_from_eyes', ' stomach_pain', ' acidity',
       ...
       ' bladder_discomfort', ' foul_smell_of urine',
       ' continuous_feel_of_urine', ' skin_peeling', ' silver_like_dusting',
       ' small_dents_in_nails', ' inflammatory_nails', ' blister',
       ' red_sore_around_nose', ' yellow_crust_ooze'],
      dtype='object', length=131)

In [34]:
# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## 1. Decision Tree Classifier

In [37]:
# Initialize the Decision Tree Classifier
dt_model = DecisionTreeClassifier(criterion='entropy', max_depth=10, random_state=42)

In [39]:
# Train the model
dt_model.fit(X_train, y_train)

DecisionTreeClassifier(criterion='entropy', max_depth=10, random_state=42)

In [41]:
# Predict on test data
y_pred = dt_model.predict(X_test)


In [43]:
y_pred

array(['Jaundice', 'Hepatitis B', 'Hypertension ', 'Hyperthyroidism',
       'Hepatitis C', 'Allergy', 'Dengue', 'Allergy', 'Hepatitis C',
       'Allergy', 'Allergy', 'Malaria', 'Hepatitis C', 'Hepatitis D',
       'Typhoid', 'Diabetes ', 'Dengue', 'Tuberculosis', 'Hypothyroidism',
       'hepatitis A', 'Hepatitis C', 'Gastroenteritis', 'Allergy',
       'Drug Reaction', 'Hepatitis B', 'Allergy', 'Gastroenteritis',
       'Allergy', 'Varicose veins', 'Chicken pox', 'hepatitis A', 'Acne',
       'Pneumonia', 'Bronchial Asthma', 'Chronic cholestasis',
       '(vertigo) Paroymsal  Positional Vertigo', 'Hypertension ',
       'Gastroenteritis', 'Gastroenteritis', 'Bronchial Asthma',
       'Jaundice', 'Migraine', 'Pneumonia', 'Hepatitis B', 'Allergy',
       'Allergy', 'Allergy', 'Migraine', 'Fungal infection',
       'Varicose veins', 'Hypoglycemia', 'Hypoglycemia', 'Jaundice',
       'Allergy', 'Fungal infection', 'Hypothyroidism',
       'Dimorphic hemmorhoids(piles)', 'Gastroenteritis

In [45]:
# Evaluation
print("Decision Tree Classifier Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

Decision Tree Classifier Accuracy: 0.5245901639344263

Classification Report:
                                          precision    recall  f1-score   support

(vertigo) Paroymsal  Positional Vertigo       1.00      0.50      0.67         2
                                   AIDS       0.00      0.00      0.00         1
                                   Acne       1.00      0.33      0.50         3
                    Alcoholic hepatitis       0.00      0.00      0.00         1
                                Allergy       0.27      1.00      0.43         3
                       Bronchial Asthma       0.50      0.33      0.40         3
                   Cervical spondylosis       0.00      0.00      0.00         1
                            Chicken pox       1.00      0.33      0.50         3
                    Chronic cholestasis       1.00      0.50      0.67         2
                                 Dengue       1.00      0.67      0.80         3
                             

C:\Users\kalya\anaconda3New\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\kalya\anaconda3New\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\kalya\anaconda3New\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\kalya\anaconda3New\Lib\site-pa

In [46]:
# Example prediction (symptom input should match dataset format)
example_input = X_test.iloc[1].values.reshape(1, -1)
predicted_disease = dt_model.predict(example_input)

print("\nExample symptom for testing", example_input)
print("\nPredicted disease for sample input:", predicted_disease[0])


Example symptom for testing [[1 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 1 1 1 0 0 0 0 0 0 0 0 0 1 0 0 1 0 0 0 0
  0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 1 0 0 0 0 0 0 0 1 1
  0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
  0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]]

Predicted disease for sample input: Hepatitis B


C:\Users\kalya\anaconda3New\Lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but DecisionTreeClassifier was fitted with feature names
  warnings.warn(


## 2. Random Forest Classifier

In [49]:
df

,Disease,itching,skin_rash,nodal_skin_eruptions,dischromic _patches,continuous_sneezing,shivering,chills,watering_from_eyes,stomach_pain,...,bladder_discomfort,foul_smell_of urine,continuous_feel_of_urine,skin_peeling,silver_like_dusting,small_dents_in_nails,inflammatory_nails,blister,red_sore_around_nose,yellow_crust_ooze
0,Fungal infection,1,1,1,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,Fungal infection,0,1,1,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,Fungal infection,1,0,1,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,Fungal infection,1,1,0,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,Fungal infection,1,1,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
299,Impetigo,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,1,1,1
300,Impetigo,0,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,1,1,1
301,Impetigo,0,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,1
302,Impetigo,0,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,1


In [51]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Load dataset
df = pd.read_csv("binary_symptom_dataset.csv")

# Encode target label
le = LabelEncoder()
df['Disease'] = le.fit_transform(df['Disease'])

# Split features and target
X = df.drop('Disease', axis=1)
y = df['Disease']

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Random Forest model
rf = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    class_weight="balanced"
)

# Train the model
rf.fit(X_train, y_train)

# Predictions
y_pred = rf.predict(X_test)

# Evaluation
print("Random Forest Accuracy:", accuracy_score(y_test, y_pred))
print("\Classification Report:\n", classification_report(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))

# Decode predictions back to disease names (optional)
print("\nSample Predictions (Disease Names):")
print(le.inverse_transform(y_pred[:10]))


<>:38: SyntaxWarning: invalid escape sequence '\C'
<>:38: SyntaxWarning: invalid escape sequence '\C'
C:\Users\kalya\AppData\Local\Temp\ipykernel_22028\1303925364.py:38: SyntaxWarning: invalid escape sequence '\C'
  print("\Classification Report:\n", classification_report(y_test, y_pred))


Random Forest Accuracy: 1.0
\Classification Report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00         1
           1       1.00      1.00      1.00         1
           2       1.00      1.00      1.00         1
           3       1.00      1.00      1.00         2
           4       1.00      1.00      1.00         1
           5       1.00      1.00      1.00         1
           6       1.00      1.00      1.00         1
           7       1.00      1.00      1.00         1
           8       1.00      1.00      1.00         2
           9       1.00      1.00      1.00         2
          10       1.00      1.00      1.00         2
          11       1.00      1.00      1.00         2
          12       1.00      1.00      1.00         2
          13       1.00      1.00      1.00         1
          14       1.00      1.00      1.00         1
          15       1.00      1.00      1.00         1
          16       1.00     

## 3. Naive Bayes

In [53]:
# bernoulli_naive_bayes_disease_prediction.py

import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold, GridSearchCV
from sklearn.naive_bayes import BernoulliNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
import joblib   # for saving model

# ------------------------
# 1. Load dataset
# ------------------------
#df = pd.read_csv("binary_symptom_dataset.csv")
#print("Dataset shape:", df.shape)
#print("Target value counts:\n", df['Disease'].value_counts())

# ------------------------
# 2. Encode target
# ------------------------
le = LabelEncoder()
df['Disease_lbl'] = le.fit_transform(df['Disease'])

# ------------------------
# 3. Split features / target
# ------------------------
X = df.drop(columns=['Disease', 'Disease_lbl'])
y = df['Disease_lbl']

# ------------------------
# 4. Train / test split
# ------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)


print(f"Train shape: {X_train.shape}, Test shape: {X_test.shape}")

# ------------------------
# 5. Build BernoulliNB model
# ------------------------
bnb = BernoulliNB()

# Optional: quick cross-validation to estimate performance
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(bnb, X_train, y_train, cv=cv, scoring='f1_weighted', n_jobs=-1)
print("Cross-val F1-weighted scores:", np.round(cv_scores, 4))
print("Mean CV F1-weighted:", np.round(cv_scores.mean(), 4))

# ------------------------
# 6. Hyperparameter tuning (optional)
#    Here we tune the smoothing parameter 'alpha'
# ------------------------
param_grid = {'alpha': [0.01, 0.1, 0.5, 1.0]}
grid = GridSearchCV(BernoulliNB(), param_grid, cv=cv, scoring='f1_weighted', n_jobs=-1)
grid.fit(X_train, y_train)
print("Best alpha:", grid.best_params_)
best_bnb = grid.best_estimator_

# ------------------------
# 7. Train final model on training data
# ------------------------
best_bnb.fit(X_train, y_train)

# ------------------------
# 8. Predict & evaluate on test set
# ------------------------
y_pred = best_bnb.predict(X_test)
y_proba = best_bnb.predict_proba(X_test)  # probabilities per class

print("\nNaive Bayes Test Accuracy:", accuracy_score(y_test, y_pred))
print("\nWeighted F1:", f1_score(y_test, y_pred, average='weighted'))
# Convert encoded labels back to string names (decoder expects encoded ints)

labels = np.unique(y_test)

target_names = [str(name) for name in le.inverse_transform(labels)]

print(
    "\nClassification Report:\n",
    classification_report(
        y_test,
        y_pred,
        labels=labels,
        target_names=target_names
    )
)
class_names = le.inverse_transform(labels)

#print("\nClassification Report:\n", classification_report(y_test, y_pred, target_names=class_names))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))

# ------------------------
# 9. Show sample predictions (decoded)
# ------------------------
sample_idx = np.arange(min(10, len(y_pred)))
pred_names = le.inverse_transform(y_pred[sample_idx])
true_names = le.inverse_transform(y_test.iloc[sample_idx])
print("\nSample predictions (first {} test samples):".format(len(sample_idx)))
for i, idx in enumerate(sample_idx):
    probs = y_proba[idx]
    top3 = np.argsort(probs)[-3:][::-1]  # top 3 class indices
    top3_names = le.inverse_transform(top3)
    top3_probs = probs[top3]
    print(f"Sample {i}: True = {true_names[i]}, Pred = {pred_names[i]}, Top3 = {list(zip(top3_names, np.round(top3_probs,3)))}")

# ------------------------
# 10. Save model + label encoder
# ------------------------
joblib.dump(best_bnb, "bernoulli_nb_model.joblib")
joblib.dump(le, "label_encoder.joblib")
print("\nSaved model -> bernoulli_nb_model.joblib and label encoder -> label_encoder.joblib")


Train shape: (243, 131), Test shape: (61, 131)


C:\Users\kalya\anaconda3New\Lib\site-packages\sklearn\model_selection\_split.py:776: UserWarning: The least populated class in y has only 4 members, which is less than n_splits=5.
  warnings.warn(


Cross-val F1-weighted scores: [0.8946 0.8129 0.8483 0.8646 0.9514]
Mean CV F1-weighted: 0.8744


C:\Users\kalya\anaconda3New\Lib\site-packages\sklearn\model_selection\_split.py:776: UserWarning: The least populated class in y has only 4 members, which is less than n_splits=5.
  warnings.warn(


Best alpha: {'alpha': 0.01}

Naive Bayes Test Accuracy: 0.9836065573770492

Weighted F1: 0.9825136612021859

Classification Report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00         1
           1       1.00      1.00      1.00         1
           2       1.00      1.00      1.00         1
           3       1.00      1.00      1.00         2
           4       1.00      1.00      1.00         1
           5       1.00      1.00      1.00         1
           6       1.00      1.00      1.00         1
           7       1.00      1.00      1.00         1
           8       1.00      1.00      1.00         2
           9       1.00      0.50      0.67         2
          10       1.00      1.00      1.00         2
          11       1.00      1.00      1.00         2
          12       1.00      1.00      1.00         2
          13       1.00      1.00      1.00         1
          14       1.00      1.00      1.00         1
  

## 4. Logistic Regression

In [55]:
# logistic_regression_disease_prediction.py
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
import joblib

# ------------------------
# 1. Load dataset
# ------------------------
#df = pd.read_csv("binary_symptom_dataset.csv")
print("Dataset shape:", df.shape)
print("Target value counts:\n", df['Disease'].value_counts())

# ------------------------
# 2. Encode target labels
# ------------------------
le = LabelEncoder()
df['Disease_lbl'] = le.fit_transform(df['Disease'])

# ------------------------
# 3. Split features / target
# ------------------------
X = df.drop(columns=['Disease', 'Disease_lbl'])
y = df['Disease_lbl']

# ------------------------
# 4. Train / test split
# ------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print(f"Train shape: {X_train.shape}, Test shape: {X_test.shape}")

# ------------------------
# 5. Build a Pipeline: Scaler + LogisticRegression
#    - solver='saga' supports l1 and l2 with multinomial
#    - multi_class='multinomial' for true multiclass
# ------------------------
pipe = Pipeline([
    ("scaler", StandardScaler()),             # scale features
    ("clf", LogisticRegression(
        solver="saga",
        multi_class="multinomial",
        max_iter=1000,
        random_state=42,
        n_jobs=-1
    ))
])

# ------------------------
# 6. Cross-validation baseline (optional)
# ------------------------
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
baseline_scores = cross_val_score(pipe, X_train, y_train, cv=cv, scoring='f1_weighted', n_jobs=-1)
print("Baseline CV F1-weighted (no tuning):", np.round(baseline_scores, 4))
print("Mean baseline F1-weighted:", np.round(baseline_scores.mean(), 4))

# ------------------------
# 7. Hyperparameter tuning (GridSearch)
#    Tune C (inverse regularization strength) and penalty (l1/l2).
# ------------------------
param_grid = {
    "clf__C": [0.01, 0.1, 1.0, 10.0],
    "clf__penalty": ["l1", "l2"]
}

grid = GridSearchCV(
    estimator=pipe,
    param_grid=param_grid,
    scoring="f1_weighted",
    cv=cv,
    n_jobs=-1,
    verbose=1
)

grid.fit(X_train, y_train)
print("Best params:", grid.best_params_)
print("Best CV score:", grid.best_score_)

best_model = grid.best_estimator_

# ------------------------
# 8. Evaluate on test set
# ------------------------
y_pred = best_model.predict(X_test)
y_proba = best_model.predict_proba(X_test)

print("\nLogistic Regression Test Accuracy:", accuracy_score(y_test, y_pred))
print("Test Weighted F1:", f1_score(y_test, y_pred, average="weighted"))

# For readable classification report we decode labels present in test set
labels_present = np.unique(y_test)
#target_names = le.inverse_transform(labels_present)

target_names = [
    str(name) for name in le.inverse_transform(labels_present)
]

print("\nClassification Report:\n", classification_report(y_test, y_pred, labels=labels_present, target_names=target_names))

print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))

# ------------------------
# 9. Show a few sample predictions with top-3 probabilities
# ------------------------
n_show = min(10, len(y_pred))
for i in range(n_show):
    probs = y_proba[i]
    top3_idx = np.argsort(probs)[-3:][::-1]
    top3_names = le.inverse_transform(top3_idx)
    top3_probs = probs[top3_idx]
    print(f"Sample {i}: True = {le.inverse_transform([y_test.iloc[i]])[0]}, Pred = {le.inverse_transform([y_pred[i]])[0]}, Top3 = {list(zip(top3_names, np.round(top3_probs,3)))}")

# ------------------------
# 10. Save model and label encoder
# ------------------------
joblib.dump(best_model, "logistic_regression_pipeline.joblib")
joblib.dump(le, "label_encoder.joblib")
print("\nSaved pipeline -> logistic_regression_pipeline.joblib")
print("Saved label encoder -> label_encoder.joblib")


Dataset shape: (304, 133)
Target value counts:
 Disease
21    10
11    10
8     10
30    10
19     9
25     9
10     9
36     9
22     9
40     9
37     9
24     9
28     9
12     9
34     9
39     8
29     8
26     8
3      8
9      8
31     7
6      7
0      7
20     7
33     7
35     7
16     7
5      6
27     6
13     6
7      6
23     6
14     6
18     5
4      5
32     5
17     5
1      5
2      5
38     5
15     5
Name: count, dtype: int64
Train shape: (243, 131), Test shape: (61, 131)


C:\Users\kalya\anaconda3New\Lib\site-packages\sklearn\model_selection\_split.py:776: UserWarning: The least populated class in y has only 4 members, which is less than n_splits=5.
  warnings.warn(


Baseline CV F1-weighted (no tuning): [1.     1.     0.9796 1.     1.    ]
Mean baseline F1-weighted: 0.9959
Fitting 5 folds for each of 8 candidates, totalling 40 fits


C:\Users\kalya\anaconda3New\Lib\site-packages\sklearn\model_selection\_split.py:776: UserWarning: The least populated class in y has only 4 members, which is less than n_splits=5.
  warnings.warn(
C:\Users\kalya\anaconda3New\Lib\site-packages\sklearn\linear_model\_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


Best params: {'clf__C': 0.1, 'clf__penalty': 'l2'}
Best CV score: 0.9959183673469388

Logistic Regression Test Accuracy: 1.0
Test Weighted F1: 1.0

Classification Report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00         1
           1       1.00      1.00      1.00         1
           2       1.00      1.00      1.00         1
           3       1.00      1.00      1.00         2
           4       1.00      1.00      1.00         1
           5       1.00      1.00      1.00         1
           6       1.00      1.00      1.00         1
           7       1.00      1.00      1.00         1
           8       1.00      1.00      1.00         2
           9       1.00      1.00      1.00         2
          10       1.00      1.00      1.00         2
          11       1.00      1.00      1.00         2
          12       1.00      1.00      1.00         2
          13       1.00      1.00      1.00         1
          14     

## 5.Support Vector Machine

In [58]:
# svm_disease_prediction.py
# -----------------------------------------
# SVM pipeline for multiclass disease prediction
# Suitable for binary symptom features (0/1)
# -----------------------------------------

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
import joblib

# ------------------------
# 1. Load dataset
# ------------------------
# The CSV should have a 'Disease' column (string labels) and the rest are binary symptom columns.
#df = pd.read_csv("binary_symptom_dataset.csv")
print("Dataset shape:", df.shape)
print("Example target counts:\n", df['Disease'].value_counts())

# ------------------------
# 2. Encode target labels
# ------------------------
# Convert string disease names to integer labels for model training.
le = LabelEncoder()
df['Disease_lbl'] = le.fit_transform(df['Disease'])

# ------------------------
# 3. Prepare features (X) and target (y)
# ------------------------
# Drop the original text label and the encoded label column stays as y.
X = df.drop(columns=['Disease', 'Disease_lbl'])
y = df['Disease_lbl']

# ------------------------
# 4. Train / test split
# ------------------------
# Use stratify=y to preserve class distribution in train/test sets.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print(f"Train shape: {X_train.shape}, Test shape: {X_test.shape}")

# ------------------------
# 5. Build SVM pipeline
# ------------------------
# StandardScaler normalises features which is important for SVM performance.
# SVC(probability=True) allows predict_proba but increases training time and memory.
pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("svc", SVC(probability=True, class_weight="balanced", random_state=42))
])

# ------------------------
# 6. Baseline cross-validation (optional)
# ------------------------
# Quick baseline using stratified 5-fold CV to estimate performance before tuning.
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
baseline_scores = cross_val_score(pipe, X_train, y_train, cv=cv, scoring='f1_weighted', n_jobs=-1)
print("Baseline CV F1-weighted (no tuning):", np.round(baseline_scores, 4))
print("Mean baseline F1-weighted:", np.round(baseline_scores.mean(), 4))

# ------------------------
# 7. Hyperparameter tuning using GridSearchCV
# ------------------------
# We tune C (regularization), kernel, and gamma (for rbf/poly kernels).
# Note: grid size affects runtime. Start with a small grid and expand if needed.
param_grid = {
    "svc__C": [0.1, 1.0, 10.0],                 # regularization strength
    "svc__kernel": ["linear", "rbf"],          # kernel types: linear is faster, rbf handles non-linear
    "svc__gamma": ["scale", "auto"]            # kernel coefficient for 'rbf'
}

grid = GridSearchCV(
    estimator=pipe,
    param_grid=param_grid,
    scoring="f1_weighted",
    cv=cv,
    n_jobs=-1,
    verbose=2
)

# Fit GridSearch on training data (this may take time)
grid.fit(X_train, y_train)

print("Best params found:", grid.best_params_)
print("Best CV score:", grid.best_score_)

# Get the best pipeline (includes fitted scaler + SVC)
best_model = grid.best_estimator_

# ------------------------
# 8. Evaluate on test set
# ------------------------
y_pred = best_model.predict(X_test)
y_proba = best_model.predict_proba(X_test)  # probabilities for each class (requires probability=True)

print("\nSVM Classifier Test Accuracy:", accuracy_score(y_test, y_pred))
print("Test Weighted F1:", f1_score(y_test, y_pred, average="weighted"))

# For a readable report, decode only the labels present in the test set
labels_present = np.unique(y_test)
#target_names = le.inverse_transform(labels_present)
target_names = [
    str(name) for name in le.inverse_transform(labels_present)
]
print("\nClassification Report:\n",
      classification_report(y_test, y_pred, labels=labels_present, target_names=target_names))

print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))

# ------------------------
# 9. Show a few sample predictions with top-3 probabilities
# ------------------------
n_show = min(10, len(y_pred))
for i in range(n_show):
    probs = y_proba[i]
    top3_idx = np.argsort(probs)[-3:][::-1]          # indices of top-3 predicted classes
    top3_names = le.inverse_transform(top3_idx)      # convert encoded indices to disease names
    top3_probs = probs[top3_idx]
    true_name = le.inverse_transform([y_test.iloc[i]])[0]
    pred_name = le.inverse_transform([y_pred[i]])[0]
    print(f"Sample {i}: True = {true_name}, Pred = {pred_name}, Top3 = {list(zip(top3_names, np.round(top3_probs,3)))}")

# ------------------------
# 10. Save the best model and label encoder for later use
# ------------------------
joblib.dump(best_model, "svm_pipeline_best.joblib")
joblib.dump(le, "label_encoder.joblib")
print("\nSaved pipeline -> svm_pipeline_best.joblib")
print("Saved label encoder -> label_encoder.joblib")

# -----------------------------------------
# End of SVM pipeline
# -----------------------------------------


Dataset shape: (304, 133)
Example target counts:
 Disease
21    10
11    10
8     10
30    10
19     9
25     9
10     9
36     9
22     9
40     9
37     9
24     9
28     9
12     9
34     9
39     8
29     8
26     8
3      8
9      8
31     7
6      7
0      7
20     7
33     7
35     7
16     7
5      6
27     6
13     6
7      6
23     6
14     6
18     5
4      5
32     5
17     5
1      5
2      5
38     5
15     5
Name: count, dtype: int64
Train shape: (243, 131), Test shape: (61, 131)
Baseline CV F1-weighted (no tuning): [1.     1.     0.9796 0.9722 0.9722]
Mean baseline F1-weighted: 0.9848
Fitting 5 folds for each of 12 candidates, totalling 60 fits


C:\Users\kalya\anaconda3New\Lib\site-packages\sklearn\model_selection\_split.py:776: UserWarning: The least populated class in y has only 4 members, which is less than n_splits=5.
  warnings.warn(
C:\Users\kalya\anaconda3New\Lib\site-packages\sklearn\model_selection\_split.py:776: UserWarning: The least populated class in y has only 4 members, which is less than n_splits=5.
  warnings.warn(


Best params found: {'svc__C': 1.0, 'svc__gamma': 'scale', 'svc__kernel': 'rbf'}
Best CV score: 0.9848072562358278

SVM Classifier Test Accuracy: 1.0
Test Weighted F1: 1.0

Classification Report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00         1
           1       1.00      1.00      1.00         1
           2       1.00      1.00      1.00         1
           3       1.00      1.00      1.00         2
           4       1.00      1.00      1.00         1
           5       1.00      1.00      1.00         1
           6       1.00      1.00      1.00         1
           7       1.00      1.00      1.00         1
           8       1.00      1.00      1.00         2
           9       1.00      1.00      1.00         2
          10       1.00      1.00      1.00         2
          11       1.00      1.00      1.00         2
          12       1.00      1.00      1.00         2
          13       1.00      1.00      1.00    

## 6. K-Nearest neighbour regression

In [60]:
# knn_disease_prediction.py
# ---------------------------------------------------
# K-Nearest Neighbours for multiclass disease prediction
# Works with binary symptom dataset (0/1 features)
# ---------------------------------------------------

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
import joblib

# ------------------------
# 1. Load dataset
# ------------------------
# Dataset contains:
# - 'Disease' column (target)
# - Remaining columns are binary symptoms (0/1)
#df = pd.read_csv("binary_symptom_dataset.csv")

print("Dataset shape:", df.shape)
print("\nDisease distribution:\n", df['Disease'].value_counts())

# ------------------------
# 2. Encode target labels
# ------------------------
# Convert disease names (strings) into numeric labels
le = LabelEncoder()
df['Disease_lbl'] = le.fit_transform(df['Disease'])

# ------------------------
# 3. Split features and target
# ------------------------
X = df.drop(columns=['Disease', 'Disease_lbl'])
y = df['Disease_lbl']

# ------------------------
# 4. Train / test split
# ------------------------
# Stratified split preserves disease distribution
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print(f"\nTrain shape: {X_train.shape}, Test shape: {X_test.shape}")

# ------------------------
# 5. Build KNN pipeline
# ------------------------
# Scaling is CRITICAL for KNN because it is distance-based
pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier())
])

# ------------------------
# 6. Baseline cross-validation (optional)
# ------------------------
# Estimate performance before tuning hyperparameters
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

baseline_scores = cross_val_score(
    pipe,
    X_train,
    y_train,
    cv=cv,
    scoring="f1_weighted",
    n_jobs=-1
)

print("\nBaseline CV F1-weighted scores:", np.round(baseline_scores, 4))
print("Mean baseline CV F1-weighted:", np.round(baseline_scores.mean(), 4))

# ------------------------
# 7. Hyperparameter tuning (GridSearchCV)
# ------------------------
# Tune:
# - n_neighbors (K)
# - weights (uniform vs distance)
# - distance metric (p=1 -> Manhattan, p=2 -> Euclidean)
param_grid = {
    "knn__n_neighbors": [3, 5, 7, 9, 11],
    "knn__weights": ["uniform", "distance"],
    "knn__p": [1, 2]
}

grid = GridSearchCV(
    estimator=pipe,
    param_grid=param_grid,
    scoring="f1_weighted",
    cv=cv,
    n_jobs=-1,
    verbose=2
)

grid.fit(X_train, y_train)

print("\nBest parameters found:", grid.best_params_)
print("Best CV F1-weighted score:", grid.best_score_)

best_model = grid.best_estimator_

# ------------------------
# 8. Evaluate on test set
# ------------------------
y_pred = best_model.predict(X_test)

print("\nK-Nearest Neighbour Test Accuracy:", accuracy_score(y_test, y_pred))
print("Test Weighted F1:", f1_score(y_test, y_pred, average="weighted"))

# Decode only labels present in test set for readability
labels_present = np.unique(y_test)
#target_names = le.inverse_transform(labels_present)
target_names = [
    str(name) for name in le.inverse_transform(labels_present)
]

print("\nClassification Report:\n",
      classification_report(y_test, y_pred, labels=labels_present, target_names=target_names))

print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))

# ------------------------
# 9. Save trained model and encoder
# ------------------------
joblib.dump(best_model, "knn_pipeline_best.joblib")
joblib.dump(le, "label_encoder.joblib")

print("\nSaved model -> knn_pipeline_best.joblib")
print("Saved label encoder -> label_encoder.joblib")

# ---------------------------------------------------
# End of KNN pipeline
# ---------------------------------------------------


Dataset shape: (304, 133)

Disease distribution:
 Disease
21    10
11    10
8     10
30    10
19     9
25     9
10     9
36     9
22     9
40     9
37     9
24     9
28     9
12     9
34     9
39     8
29     8
26     8
3      8
9      8
31     7
6      7
0      7
20     7
33     7
35     7
16     7
5      6
27     6
13     6
7      6
23     6
14     6
18     5
4      5
32     5
17     5
1      5
2      5
38     5
15     5
Name: count, dtype: int64

Train shape: (243, 131), Test shape: (61, 131)


C:\Users\kalya\anaconda3New\Lib\site-packages\sklearn\model_selection\_split.py:776: UserWarning: The least populated class in y has only 4 members, which is less than n_splits=5.
  warnings.warn(



Baseline CV F1-weighted scores: [0.9728 0.9714 0.9524 0.9444 1.    ]
Mean baseline CV F1-weighted: 0.9682
Fitting 5 folds for each of 20 candidates, totalling 100 fits


C:\Users\kalya\anaconda3New\Lib\site-packages\sklearn\model_selection\_split.py:776: UserWarning: The least populated class in y has only 4 members, which is less than n_splits=5.
  warnings.warn(



Best parameters found: {'knn__n_neighbors': 3, 'knn__p': 1, 'knn__weights': 'distance'}
Best CV F1-weighted score: 1.0

K-Nearest Neighbour Test Accuracy: 1.0
Test Weighted F1: 1.0

Classification Report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00         1
           1       1.00      1.00      1.00         1
           2       1.00      1.00      1.00         1
           3       1.00      1.00      1.00         2
           4       1.00      1.00      1.00         1
           5       1.00      1.00      1.00         1
           6       1.00      1.00      1.00         1
           7       1.00      1.00      1.00         1
           8       1.00      1.00      1.00         2
           9       1.00      1.00      1.00         2
          10       1.00      1.00      1.00         2
          11       1.00      1.00      1.00         2
          12       1.00      1.00      1.00         2
          13       1.00      1.00   

## 7. XGBoost

In [63]:
!pip install xgboost

# xgboost_disease_prediction.py
# ---------------------------------------------------
# XGBoost for multiclass disease prediction
# Dataset: binary_symptom_dataset.csv
# ---------------------------------------------------

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
import joblib

from xgboost import XGBClassifier

# ------------------------
# 1. Load dataset
# ------------------------
#df = pd.read_csv("binary_symptom_dataset.csv")

print("Dataset shape:", df.shape)
print("\nDisease distribution:\n", df['Disease'].value_counts())

# ------------------------
# 2. Encode target labels
# ------------------------
# XGBoost requires numeric class labels
le = LabelEncoder()
df['Disease_lbl'] = le.fit_transform(df['Disease'])

# ------------------------
# 3. Split features and target
# ------------------------
X = df.drop(columns=['Disease', 'Disease_lbl'])
y = df['Disease_lbl']

num_classes = y.nunique()
print("\nNumber of classes:", num_classes)

# ------------------------
# 4. Train / test split
# ------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print(f"Train shape: {X_train.shape}, Test shape: {X_test.shape}")

# ------------------------
# 5. Define base XGBoost model
# ------------------------
# multi:softprob -> returns class probabilities
# eval_metric='mlogloss' is standard for multiclass classification
xgb_base = XGBClassifier(
    objective="multi:softprob",
    num_class=num_classes,
    eval_metric="mlogloss",
    use_label_encoder=False,
    random_state=42,
    n_jobs=-1
)

# ------------------------
# 6. Baseline cross-validation (optional)
# ------------------------
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

baseline_scores = cross_val_score(
    xgb_base,
    X_train,
    y_train,
    cv=cv,
    scoring="f1_weighted",
    n_jobs=-1
)

print("\nBaseline CV F1-weighted scores:", np.round(baseline_scores, 4))
print("Mean baseline CV F1-weighted:", np.round(baseline_scores.mean(), 4))

# ------------------------
# 7. Hyperparameter tuning (GridSearchCV)
# ------------------------
# Keep grid reasonable to avoid long runtimes
param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [3, 5, 7],
    "learning_rate": [0.05, 0.1],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.8, 1.0]
}

grid = GridSearchCV(
    estimator=xgb_base,
    param_grid=param_grid,
    scoring="f1_weighted",
    cv=cv,
    n_jobs=-1,
    verbose=2
)

grid.fit(X_train, y_train)

print("\nBest parameters found:", grid.best_params_)
print("Best CV F1-weighted score:", grid.best_score_)

best_model = grid.best_estimator_

# ------------------------
# 8. Evaluate on test set
# ------------------------
y_pred = best_model.predict(X_test)
y_proba = best_model.predict_proba(X_test)

print("\nXGBoost Test Accuracy:", accuracy_score(y_test, y_pred))
print("Test Weighted F1:", f1_score(y_test, y_pred, average="weighted"))

# Decode only labels present in test set
labels_present = np.unique(y_test)
#target_names = le.inverse_transform(labels_present)
target_names = [
    str(name) for name in le.inverse_transform(labels_present)
]

print("\nClassification Report:\n",
      classification_report(y_test, y_pred, labels=labels_present, target_names=target_names))

print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))

# ------------------------
# 9. Show sample predictions with top-3 probabilities
# ------------------------
n_show = min(10, len(y_pred))
for i in range(n_show):
    probs = y_proba[i]
    top3_idx = np.argsort(probs)[-3:][::-1]
    top3_names = le.inverse_transform(top3_idx)
    top3_probs = probs[top3_idx]

    true_name = le.inverse_transform([y_test.iloc[i]])[0]
    pred_name = le.inverse_transform([y_pred[i]])[0]

    print(f"Sample {i}: True = {true_name}, Pred = {pred_name}, "
          f"Top3 = {list(zip(top3_names, np.round(top3_probs, 3)))}")

# ------------------------
# 10. Save trained model and label encoder
# ------------------------
joblib.dump(best_model, "xgboost_model_best.joblib")
joblib.dump(le, "label_encoder.joblib")

print("\nSaved model -> xgboost_model_best.joblib")
print("Saved label encoder -> label_encoder.joblib")

# ---------------------------------------------------
# End of XGBoost pipeline
# ---------------------------------------------------


Dataset shape: (304, 133)

Disease distribution:
 Disease
21    10
11    10
8     10
30    10
19     9
25     9
10     9
36     9
22     9
40     9
37     9
24     9
28     9
12     9
34     9
39     8
29     8
26     8
3      8
9      8
31     7
6      7
0      7
20     7
33     7
35     7
16     7
5      6
27     6
13     6
7      6
23     6
14     6
18     5
4      5
32     5
17     5
1      5
2      5
38     5
15     5
Name: count, dtype: int64

Number of classes: 41
Train shape: (243, 131), Test shape: (61, 131)


C:\Users\kalya\anaconda3New\Lib\site-packages\sklearn\model_selection\_split.py:776: UserWarning: The least populated class in y has only 4 members, which is less than n_splits=5.
  warnings.warn(



Baseline CV F1-weighted scores: [0.8844 0.8537 0.8966 0.8424 0.8944]
Mean baseline CV F1-weighted: 0.8743
Fitting 5 folds for each of 48 candidates, totalling 240 fits


C:\Users\kalya\anaconda3New\Lib\site-packages\sklearn\model_selection\_split.py:776: UserWarning: The least populated class in y has only 4 members, which is less than n_splits=5.
  warnings.warn(
C:\Users\kalya\anaconda3New\Lib\site-packages\xgboost\core.py:158: UserWarning: [17:26:51] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-08cbc0333d8d4aae1-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)



Best parameters found: {'colsample_bytree': 0.8, 'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 200, 'subsample': 0.8}
Best CV F1-weighted score: 0.9533560090702947

XGBoost Test Accuracy: 1.0
Test Weighted F1: 1.0

Classification Report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00         1
           1       1.00      1.00      1.00         1
           2       1.00      1.00      1.00         1
           3       1.00      1.00      1.00         2
           4       1.00      1.00      1.00         1
           5       1.00      1.00      1.00         1
           6       1.00      1.00      1.00         1
           7       1.00      1.00      1.00         1
           8       1.00      1.00      1.00         2
           9       1.00      1.00      1.00         2
          10       1.00      1.00      1.00         2
          11       1.00      1.00      1.00         2
          12       1.00      1.00      1.00      

In [64]:
# Ensure Disease is string
df['Disease'] = df['Disease'].astype(str)

# Encode labels
le = LabelEncoder()
df['Disease_lbl'] = le.fit_transform(df['Disease'])

X = df.drop(columns=['Disease', 'Disease_lbl'])
y = df['Disease_lbl']

# Normalize feature names BEFORE training
X.columns = [normalize_feature(c) for c in X.columns]
FEATURE_COLUMNS = X.columns.tolist()

# Train final model
final_xgb_model.fit(X, y)

# Save artifacts
CLASS_NAMES = le.classes_.astype(str)

print(CLASS_NAMES[:10])
print(type(CLASS_NAMES[0]))


joblib.dump(final_xgb_model, "xgboost_final_full_data_model.joblib")
joblib.dump(CLASS_NAMES, "xgboost_class_names.joblib")
joblib.dump(FEATURE_COLUMNS, "xgboost_feature_columns.joblib")


NameError: name 'normalize_feature' is not defined